# LeetCode 2888 — Reshape Data: Concatenate bằng Python (pandas)

Link đề: https://leetcode.com/problems/reshape-data-concatenate/ · Độ khó: **Easy** · Chủ đề: Pandas (30 Days of Pandas)

Skeleton LeetCode (không có `class Solution`; tên hàm / chữ ký: đối chiếu lại với đề gốc):

```python
import pandas as pd

def concatenateTables(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    ...
```

Ghi chú: dữ liệu ví dụ trong notebook được viết theo trí nhớ về đề; nên đối chiếu lại với đề gốc nếu muốn khớp từng ô.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có hai bảng `df1`, `df2` **cùng các cột** (`student_id`, `name`, `age`); **nối `df2` xuống dưới `df1`** thành một bảng duy nhất.
2. Hiểu **trục hàng (`axis=0`)** và **trục cột (`axis=1`)**: nối dọc (thêm hàng) khác nối ngang (thêm cột).
3. Đi từ cách **vòng lặp gom từng hàng vào danh sách** → **`pd.concat([df1, df2])`** → **`ignore_index=True`** → **`to_dict("records")` rồi dựng lại bảng** → **nối mảng NumPy rồi gán lại tên cột và kiểu**.
4. Hiểu **nhãn index** sau khi nối (`0,1,2,3,0,1` nếu không đánh lại) và **cách `concat` ghép theo tên cột** (không theo vị trí).
5. Tránh các lỗi kinh điển: `df1.append(df2)` (đã bị gỡ), `pd.concat(df1, df2)` (thiếu dấu `[...]`), nhầm `axis=1`, quên đánh lại index, và nối hai bảng khác cột.

## 2. Đề bài & ví dụ

Có hai bảng `df1` và `df2` cùng có các cột `student_id`, `name`, `age`.

Hãy viết hàm **nối hai bảng theo chiều dọc**: các hàng của `df2` đứng **sau** các hàng của `df1`, và trả về bảng kết quả.

**Ví dụ**

- Input: `df1`

| student_id | name | age |
|---|---|---|
| 1 | Mason | 8 |
| 2 | Ava | 6 |
| 3 | Taylor | 15 |
| 4 | Georgia | 17 |

  và `df2`

| student_id | name | age |
|---|---|---|
| 5 | Leo | 7 |
| 6 | Alex | 7 |

- Output:

| student_id | name | age |
|---|---|---|
| 1 | Mason | 8 |
| 2 | Ava | 6 |
| 3 | Taylor | 15 |
| 4 | Georgia | 17 |
| 5 | Leo | 7 |
| 6 | Alex | 7 |

- Giải thích: bảng kết quả có `4 + 2 = 6` hàng; thứ tự: toàn bộ `df1` trước, rồi tới `df2`.

(Giá trị cụ thể trong bảng: đối chiếu lại với đề gốc. Về nhãn hàng của kết quả: notebook này dùng `ignore_index=True` để được `0..5`; việc trình chấm có so nhãn hàng hay không nên đối chiếu với đề gốc.)

In [1]:
from typing import List
import pandas as pd
import numpy as np

df1 = pd.DataFrame({
    "student_id": [1, 2, 3, 4],
    "name": ["Mason", "Ava", "Taylor", "Georgia"],
    "age": [8, 6, 15, 17],
})
df2 = pd.DataFrame({
    "student_id": [5, 6],
    "name": ["Leo", "Alex"],
    "age": [7, 7],
})
print(df1)
print()
print(df2)
print("\nhình dạng:", df1.shape, df2.shape)
print("pandas", pd.__version__)

   student_id     name  age
0           1    Mason    8
1           2      Ava    6
2           3   Taylor   15
3           4  Georgia   17

   student_id  name  age
0           5   Leo    7
1           6  Alex    7

hình dạng: (4, 3) (2, 3)
pandas 3.0.5


## 3. Kiến thức nền

### 3.1. Trục `axis=0` (hàng) và `axis=1` (cột)

Bảng có hai chiều. **`axis=0`** là chiều **đi xuống** qua các hàng; **`axis=1`** là chiều **đi ngang** qua các cột. Nối theo `axis=0` nghĩa là **xếp chồng bảng này lên bảng kia** (thêm hàng); nối theo `axis=1` là **đặt cạnh nhau** (thêm cột). Bài này cần xếp chồng nên dùng `axis=0` (mặc định).

In [2]:
print("df1.shape =", df1.shape, " -> (số hàng, số cột)")
print("hàng:", len(df1), "| cột:", len(df1.columns))
print("xếp chồng (axis=0): hàng cộng lại ->", len(df1) + len(df2), "hàng, cột giữ nguyên")

df1.shape = (4, 3)  -> (số hàng, số cột)
hàng: 4 | cột: 3
xếp chồng (axis=0): hàng cộng lại -> 6 hàng, cột giữ nguyên


### 3.2. `pd.concat([bảng1, bảng2, ...])`

`pd.concat` nhận **một danh sách** các bảng (chú ý dấu `[...]`) và **nối chúng lại**. Mặc định `axis=0` (nối dọc). Hàm trả về **bảng mới**, không sửa các bảng cũ.

In [3]:
both = pd.concat([df1, df2])
print(both)
print("hình dạng:", both.shape)
print("df1 vẫn", df1.shape, "| df2 vẫn", df2.shape)

   student_id     name  age
0           1    Mason    8
1           2      Ava    6
2           3   Taylor   15
3           4  Georgia   17
0           5      Leo    7
1           6     Alex    7
hình dạng: (6, 3)
df1 vẫn (4, 3) | df2 vẫn (2, 3)


### 3.3. Nhãn index sau khi nối

Mỗi bảng có nhãn hàng riêng (`0..3` và `0..1`). `concat` **giữ nguyên nhãn** nên kết quả có nhãn **lặp** `0, 1, 2, 3, 0, 1`. Hệ quả: `both.loc[0]` trả về **hai hàng**. Thêm `ignore_index=True` để pandas **đánh lại** nhãn `0..n-1`.

In [4]:
print(both.index.tolist())                          # [0, 1, 2, 3, 0, 1]
print(both.loc[0])                                  # nhãn 0 xuất hiện 2 lần -> 2 hàng

fixed = pd.concat([df1, df2], ignore_index=True)
print(fixed.index.tolist())                         # [0, 1, 2, 3, 4, 5]
print(both.reset_index(drop=True).equals(fixed))    # True: reset_index(drop=True) cũng được

[0, 1, 2, 3, 0, 1]
   student_id   name  age
0           1  Mason    8
0           5    Leo    7
[0, 1, 2, 3, 4, 5]
True


### 3.4. Ghép theo TÊN cột, không theo vị trí

`concat` **căn các bảng theo tên cột**. Nếu `df2` có cùng cột nhưng **khác thứ tự**, kết quả vẫn ghép đúng: cột `age` của `df2` vẫn nằm dưới cột `age` của `df1`. Nếu `df2` **thiếu** một cột, các ô tương ứng thành `NaN`. Thứ tự cột kết quả theo bảng đầu tiên.

In [5]:
df2_shuffled = df2[["age", "name", "student_id"]]            # đổi thứ tự cột
print(pd.concat([df1, df2_shuffled], ignore_index=True).tail(3))

df2_missing = df2[["student_id", "name"]]                    # thiếu cột age
print(pd.concat([df1, df2_missing], ignore_index=True).tail(3))

   student_id     name  age
3           4  Georgia   17
4           5      Leo    7
5           6     Alex    7
   student_id     name   age
3           4  Georgia  17.0
4           5      Leo   NaN
5           6     Alex   NaN


### 3.5. Nối ngang (`axis=1`) — khác hoàn toàn

`pd.concat([df1, df2], axis=1)` đặt hai bảng **cạnh nhau** và căn theo **nhãn hàng**: hàng `0` của `df1` ghép với hàng `0` của `df2`... Với bài này nó **không đúng**: ta cần thêm hàng chứ không thêm cột.

In [6]:
side = pd.concat([df1, df2], axis=1)
print(side)
print("hình dạng:", side.shape)           # (4, 6): 6 cột, 2 hàng cuối bị thiếu df2

   student_id     name  age  student_id  name  age
0           1    Mason    8         5.0   Leo  7.0
1           2      Ava    6         6.0  Alex  7.0
2           3   Taylor   15         NaN   NaN  NaN
3           4  Georgia   17         NaN   NaN  NaN
hình dạng: (4, 6)


### 3.6. `to_dict("records")` và danh sách từ điển

`df.to_dict("records")` biến mỗi hàng thành **một từ điển** `{tên_cột: giá_trị}` và trả về **danh sách các từ điển**. Ngược lại, `pd.DataFrame(danh_sách_từ_điển)` dựng lại bảng. Danh sách Python nối được bằng toán tử `+`.

In [7]:
r1 = df1.to_dict("records")
print(r1[0])                               # {'student_id': 1, 'name': 'Mason', 'age': 8}
print(len(r1), "từ điển")
merged = r1 + df2.to_dict("records")       # nối hai danh sách
print(len(merged), "từ điển sau khi nối")
print(pd.DataFrame(merged).shape)

{'student_id': 1, 'name': 'Mason', 'age': 8}
4 từ điển
6 từ điển sau khi nối
(6, 3)


### 3.7. Mảng NumPy `to_numpy()` và `np.concatenate`

`df.to_numpy()` trả **mảng NumPy 2 chiều** chứa dữ liệu của bảng. Vì bảng của ta có cả số lẫn chữ nên mảng có kiểu `object` (mọi giá trị đều là "đối tượng Python"). `np.concatenate([a, b])` nối hai mảng theo chiều dọc. Dựng lại bảng thì tên cột và **kiểu cột** phải khai báo lại.

In [8]:
a = df1.to_numpy()
print(a.dtype, a.shape)                    # object (4, 3)
stacked = np.concatenate([df1.to_numpy(), df2.to_numpy()])
print(stacked.shape)                       # (6, 3)
print(pd.DataFrame(stacked, columns=df1.columns).dtypes.to_dict())    # kiểu đã mất, thành object

object (4, 3)
(6, 3)
{'student_id': dtype('O'), 'name': <StringDtype(storage='python', na_value=nan)>, 'age': dtype('O')}


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy tưởng tượng hai tờ danh sách lớp có **cùng tiêu đề**: tờ 1 có 4 dòng, tờ 2 có 2 dòng. Ta **chép dòng của tờ 2 nối tiếp xuống cuối tờ 1**. Kết quả là một tờ 6 dòng.

| dòng | từ | student_id | name | age |
|---|---|---|---|---|
| 1 | df1 | 1 | Mason | 8 |
| 2 | df1 | 2 | Ava | 6 |
| 3 | df1 | 3 | Taylor | 15 |
| 4 | df1 | 4 | Georgia | 17 |
| 5 | df2 | 5 | Leo | 7 |
| 6 | df2 | 6 | Alex | 7 |

Thuật toán: **thu thập mọi hàng của `df1`, rồi mọi hàng của `df2`, theo đúng thứ tự, vào một bảng mới**. Hàng không bị sửa, chỉ xếp chồng.

## 5. Cách 1 — Brute Force / baseline: vòng lặp gom từng hàng

**Ý tưởng.** Tạo danh sách rỗng `rows`. Duyệt từng hàng của `df1` rồi từng hàng của `df2`, đưa mỗi hàng (dạng từ điển) vào `rows`. Cuối cùng dựng bảng từ `rows`, lấy **tên cột và kiểu cột của `df1`** để không bị lệch.

In [9]:
def concat_loop(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    rows = []                                          # nơi gom các hàng
    for _, row in df1.iterrows():                      # từng hàng của df1
        rows.append(row.to_dict())
    for _, row in df2.iterrows():                      # rồi từng hàng của df2
        rows.append(row.to_dict())
    out = pd.DataFrame(rows, columns=df1.columns)      # dựng bảng, giữ thứ tự cột của df1
    return out.astype(df1.dtypes.to_dict())            # trả lại đúng kiểu của df1


print(concat_loop(df1, df2))

   student_id     name  age
0           1    Mason    8
1           2      Ava    6
2           3   Taylor   15
3           4  Georgia   17
4           5      Leo    7
5           6     Alex    7


In [10]:
res = concat_loop(df1, df2)
print("hình dạng:", res.shape)                         # (6, 3)
print("student_id:", res["student_id"].tolist())       # [1, 2, 3, 4, 5, 6]
print("dtype:", res.dtypes.to_dict())

hình dạng: (6, 3)
student_id: [1, 2, 3, 4, 5, 6]
dtype: {'student_id': dtype('int64'), 'name': <StringDtype(storage='python', na_value=nan)>, 'age': dtype('int64')}


**Độ phức tạp (Cách 1).** Gọi `m1`, `m2` là số hàng của hai bảng, `c` là số cột. Duyệt `m1 + m2` hàng, mỗi hàng chuyển thành từ điển `O(c)`: **Time `O((m1 + m2)·c)`**; danh sách `rows` giữ mọi hàng: **Space `O((m1 + m2)·c)`**.

**Nút thắt.** `iterrows` rất chậm (tạo một `Series` cho mỗi hàng) và dễ làm đổi kiểu dữ liệu (nên phải `astype` lại). pandas có sẵn `concat`.

## 6. Ý tưởng thuật toán / bước đột phá

pandas gói đúng thao tác ta vừa làm bằng tay:

`pd.concat([df1, df2], ignore_index=True)`

- một danh sách `[df1, df2]` các bảng cần nối, **thứ tự trong danh sách = thứ tự trong kết quả**;
- `axis=0` (mặc định): xếp chồng theo hàng;
- ghép theo **tên cột**, giữ kiểu cột (hai cột `int64` nối ra `int64`);
- `ignore_index=True`: đánh lại nhãn hàng `0..n-1` (không để nhãn lặp).

Các cách khác: **không** `ignore_index` rồi `reset_index(drop=True)` (Cách 3), **`to_dict("records")` + `+`** (Cách 4), **NumPy** (Cách 5).

## 7. Các cách tiếp theo

### Cách 2 — `pd.concat([df1, df2], ignore_index=True)` (cách chuẩn)

**Giải thích.** Một lời gọi, không sửa bảng đầu vào, trả bảng mới có nhãn `0..n-1`.

In [11]:
def concat_concat(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    return pd.concat([df1, df2], ignore_index=True)


print(concat_concat(df1, df2))

   student_id     name  age
0           1    Mason    8
1           2      Ava    6
2           3   Taylor   15
3           4  Georgia   17
4           5      Leo    7
5           6     Alex    7


In [12]:
# Code nộp LeetCode cho Cách 2
import pandas as pd

def concatenateTables(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    return pd.concat([df1, df2], ignore_index=True)

**Độ phức tạp Cách 2:** **Time `O((m1 + m2)·c)`** (sao chép dữ liệu của hai bảng vào bảng mới), **Space `O((m1 + m2)·c)`** (bảng kết quả).

### Cách 3 — `pd.concat` giữ nhãn gốc, rồi `reset_index(drop=True)`

**Giải thích.** Nối không có `ignore_index` (giữ nhãn gốc: có thể dùng để biết hàng nào đến từ bảng nào), sau đó `reset_index(drop=True)` để đánh lại nhãn. Cho kết quả giống hệt Cách 2. Có thể thêm tham số `keys=["df1", "df2"]` để gắn nhãn nguồn thành một cấp index.

In [13]:
def concat_reset(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    return pd.concat([df1, df2]).reset_index(drop=True)


print(concat_reset(df1, df2))

with_keys = pd.concat([df1, df2], keys=["df1", "df2"])        # thêm cấp index chỉ nguồn
print(with_keys.index.tolist())

   student_id     name  age
0           1    Mason    8
1           2      Ava    6
2           3   Taylor   15
3           4  Georgia   17
4           5      Leo    7
5           6     Alex    7
[('df1', 0), ('df1', 1), ('df1', 2), ('df1', 3), ('df2', 0), ('df2', 1)]


In [14]:
# Code nộp LeetCode cho Cách 3
import pandas as pd

def concatenateTables(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    return pd.concat([df1, df2]).reset_index(drop=True)

**Độ phức tạp Cách 3:** **Time `O((m1 + m2)·c)`**, **Space `O((m1 + m2)·c)`**.

### Cách 4 — Nối hai danh sách từ điển (`to_dict("records")`) rồi dựng lại

**Giải thích.** Đổi mỗi bảng thành danh sách từ điển, **nối bằng toán tử `+`** của Python, rồi `pd.DataFrame(...)`. Nhanh hơn `iterrows`; nhưng với bảng lớn vẫn chậm hơn `concat` và có thể suy luận kiểu cột khác (ta `astype` lại cho chắc).

In [15]:
def concat_records(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    records = df1.to_dict("records") + df2.to_dict("records")
    return pd.DataFrame(records, columns=df1.columns).astype(df1.dtypes.to_dict())


print(concat_records(df1, df2))

   student_id     name  age
0           1    Mason    8
1           2      Ava    6
2           3   Taylor   15
3           4  Georgia   17
4           5      Leo    7
5           6     Alex    7


**Độ phức tạp Cách 4:** **Time `O((m1 + m2)·c)`** (hằng số lớn hơn `concat`), **Space `O((m1 + m2)·c)`**.

### Cách 5 — NumPy: `np.concatenate` rồi dựng lại bảng

**Giải thích.** Lấy mảng dữ liệu hai bảng bằng `to_numpy()`, nối dọc bằng `np.concatenate`, dựng bảng với `columns=df1.columns`, rồi `astype` để **phục hồi kiểu cột** (vì mảng `object` làm mất kiểu). Cách này chủ yếu để hiểu; trong thực tế dùng `concat`.

In [16]:
def concat_numpy(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    data = np.concatenate([df1.to_numpy(), df2.to_numpy()])      # mảng (m1 + m2, c)
    out = pd.DataFrame(data, columns=df1.columns)
    return out.astype(df1.dtypes.to_dict())                      # lấy lại kiểu cột


print(concat_numpy(df1, df2))

   student_id     name  age
0           1    Mason    8
1           2      Ava    6
2           3   Taylor   15
3           4  Georgia   17
4           5      Leo    7
5           6     Alex    7


**Độ phức tạp Cách 5:** **Time `O((m1 + m2)·c)`**, **Space `O((m1 + m2)·c)`**; có thêm chi phí chuyển sang/ra mảng `object`.

> Cách 2 là cách nên nộp. Nhớ `[...]` quanh hai bảng và `ignore_index=True` (hoặc `reset_index(drop=True)`).

## 8. TRACE từng bước

Ta theo dõi (a) vòng lặp tự viết, in từng hàng được đưa vào `rows`; (b) pipeline `concat` giữ nhãn rồi đánh lại.

In [17]:
def trace_loop(df1, df2):
    rows = []
    step = 0
    for source, df in (("df1", df1), ("df2", df2)):
        for label, row in df.iterrows():
            step += 1
            rows.append(row.to_dict())
            print(f"bước {step}: từ {source}, nhãn gốc={label}, hàng={row.to_dict()}  -> độ dài rows = {len(rows)}")
    out = pd.DataFrame(rows, columns=df1.columns).astype(df1.dtypes.to_dict())
    print("\ndựng bảng từ rows ->", out.shape)
    return out


_ = trace_loop(df1, df2)

bước 1: từ df1, nhãn gốc=0, hàng={'student_id': 1, 'name': 'Mason', 'age': 8}  -> độ dài rows = 1
bước 2: từ df1, nhãn gốc=1, hàng={'student_id': 2, 'name': 'Ava', 'age': 6}  -> độ dài rows = 2
bước 3: từ df1, nhãn gốc=2, hàng={'student_id': 3, 'name': 'Taylor', 'age': 15}  -> độ dài rows = 3
bước 4: từ df1, nhãn gốc=3, hàng={'student_id': 4, 'name': 'Georgia', 'age': 17}  -> độ dài rows = 4
bước 5: từ df2, nhãn gốc=0, hàng={'student_id': 5, 'name': 'Leo', 'age': 7}  -> độ dài rows = 5
bước 6: từ df2, nhãn gốc=1, hàng={'student_id': 6, 'name': 'Alex', 'age': 7}  -> độ dài rows = 6

dựng bảng từ rows -> (6, 3)


In [18]:
def trace_concat(df1, df2):
    print("BƯỚC 0 - hai bảng đầu vào:", df1.shape, df2.shape)
    print("nhãn df1:", df1.index.tolist(), "| nhãn df2:", df2.index.tolist())

    both = pd.concat([df1, df2])
    print("\nBƯỚC 1 - concat([df1, df2]) (giữ nhãn gốc):", both.shape)
    print(both)
    print("nhãn:", both.index.tolist(), "(lặp 0, 1)")

    out = both.reset_index(drop=True)
    print("\nBƯỚC 2 - reset_index(drop=True):")
    print(out)
    print("nhãn:", out.index.tolist())

    same = out.equals(pd.concat([df1, df2], ignore_index=True))
    print("\ngiống concat(..., ignore_index=True)?", same)
    return out


_ = trace_concat(df1, df2)

BƯỚC 0 - hai bảng đầu vào: (4, 3) (2, 3)
nhãn df1: [0, 1, 2, 3] | nhãn df2: [0, 1]

BƯỚC 1 - concat([df1, df2]) (giữ nhãn gốc): (6, 3)
   student_id     name  age
0           1    Mason    8
1           2      Ava    6
2           3   Taylor   15
3           4  Georgia   17
0           5      Leo    7
1           6     Alex    7
nhãn: [0, 1, 2, 3, 0, 1] (lặp 0, 1)

BƯỚC 2 - reset_index(drop=True):
   student_id     name  age
0           1    Mason    8
1           2      Ava    6
2           3   Taylor   15
3           4  Georgia   17
4           5      Leo    7
5           6     Alex    7
nhãn: [0, 1, 2, 3, 4, 5]

giống concat(..., ignore_index=True)? True


Bảng tóm tắt:

| bước | đối tượng | hình dạng / nhãn |
|---|---|---|
| 0 | `df1`, `df2` | `(4, 3)`, `(2, 3)`; nhãn `0..3`, `0..1` |
| 1 | `concat([df1, df2])` | `(6, 3)`, nhãn `[0, 1, 2, 3, 0, 1]` |
| 2 | `reset_index(drop=True)` | `(6, 3)`, nhãn `[0, 1, 2, 3, 4, 5]` |

## 9. Code nộp LeetCode

Bài pandas không có `class Solution`: chỉ cần định nghĩa hàm đúng tên và chữ ký (đối chiếu lại với đề gốc).

In [19]:
import pandas as pd

def concatenateTables(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    return pd.concat([df1, df2], ignore_index=True)


print(concatenateTables(df1, df2))

   student_id     name  age
0           1    Mason    8
1           2      Ava    6
2           3   Taylor   15
3           4  Georgia   17
4           5      Leo    7
5           6     Alex    7


## 10. Vì sao thuật toán đúng

- **Đủ hàng, đúng thứ tự:** `concat` theo `axis=0` đặt toàn bộ hàng của bảng đầu tiên, rồi tới bảng thứ hai, theo đúng thứ tự trong danh sách; nên `df1` đứng trước `df2` như đề yêu cầu.
- **Không mất, không thêm hàng:** số hàng kết quả = `len(df1) + len(df2)`; mỗi hàng gốc xuất hiện **đúng một lần**.
- **Đúng cột:** vì ghép theo **tên cột**, mỗi giá trị nằm đúng cột của nó (kể cả khi `df2` khác thứ tự cột). Với hai bảng cùng cột, không sinh `NaN`.
- **Giữ kiểu:** hai cột cùng kiểu (`int64`) nối ra cùng kiểu, nên `student_id`, `age` vẫn là số nguyên.
- **Nhãn hàng:** `ignore_index=True` thay nhãn bằng `0..n-1` nên không có nhãn lặp.
- **Tính chất kiểm được:** `result.iloc[:len(df1)]` bằng `df1` (về giá trị) và `result.iloc[len(df1):]` bằng `df2`.

## 11. Phân tích Time / Space Complexity

Gọi `m1`, `m2` là số hàng của hai bảng, `c` là số cột.

- **Time:** `O((m1 + m2)·c)`: mỗi ô của hai bảng được sao chép vào bảng kết quả đúng một lần (mã C nên hằng số nhỏ). Vòng lặp Python hằng số lớn hơn nhiều nhưng cùng bậc.
- **Space:** `O((m1 + m2)·c)` cho bảng kết quả (bảng mới, không dùng chung bộ nhớ với bảng cũ vì dữ liệu được ghép vào các khối mới).
- Không thể nhanh hơn `O((m1 + m2)·c)` vì kết quả có chừng đó ô.

## 12. Bộ test

Ta đưa các cách vào từ điển `impls`. Các hàm trên không sửa bảng đầu vào.

In [20]:
impls = {
    "loop": concat_loop,
    "concat": concat_concat,
    "reset": concat_reset,
    "records": concat_records,
    "numpy": concat_numpy,
    "submit": concatenateTables,
}

expected_example = pd.DataFrame({
    "student_id": [1, 2, 3, 4, 5, 6],
    "name": ["Mason", "Ava", "Taylor", "Georgia", "Leo", "Alex"],
    "age": [8, 6, 15, 17, 7, 7],
})

for key, fn in impls.items():
    pd.testing.assert_frame_equal(fn(df1, df2), expected_example)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [21]:
def make(ids, names=None, ages=None):
    n = len(ids)
    return pd.DataFrame({
        "student_id": pd.Series(ids, dtype="int64"),
        "name": pd.Series(names if names is not None else [f"s{i}" for i in ids], dtype="str"),
        "age": pd.Series(ages if ages is not None else [10] * n, dtype="int64"),
    })


def check_all(a, b, label):
    expected_ids = a["student_id"].tolist() + b["student_id"].tolist()
    for key, fn in impls.items():
        out = fn(a, b)
        assert out["student_id"].tolist() == expected_ids, (label, key)
        assert out.index.tolist() == list(range(len(expected_ids))), (label, key, "nhãn")
        assert out.dtypes.to_dict() == a.dtypes.to_dict(), (label, key, "dtype")
    print("PASS", label, "->", expected_ids)


check_all(make([1]), make([2]), "mỗi bảng một hàng")
check_all(make([1, 2, 3]), make([]), "df2 rỗng")
check_all(make([]), make([7, 8]), "df1 rỗng")
check_all(make([5, 4]), make([3, 2, 1]), "thứ tự không theo id (giữ nguyên thứ tự đưa vào)")
check_all(make([1, 1]), make([1, 1]), "dữ liệu trùng nhau vẫn giữ cả bốn hàng")

PASS mỗi bảng một hàng -> [1, 2]
PASS df2 rỗng -> [1, 2, 3]
PASS df1 rỗng -> [7, 8]
PASS thứ tự không theo id (giữ nguyên thứ tự đưa vào) -> [5, 4, 3, 2, 1]
PASS dữ liệu trùng nhau vẫn giữ cả bốn hàng -> [1, 1, 1, 1]


### 12.2. Hai bảng rỗng, tính chất, khác thứ tự cột, đầu vào, và phản ví dụ

In [22]:
# Hai bảng rỗng
out = concatenateTables(make([]), make([]))
assert len(out) == 0 and list(out.columns) == ["student_id", "name", "age"]
print("PASS hai bảng rỗng")

# Tính chất: kích thước, hai nửa bằng bảng gốc
out = concatenateTables(df1, df2)
assert len(out) == len(df1) + len(df2)
pd.testing.assert_frame_equal(out.iloc[:len(df1)].reset_index(drop=True), df1)
pd.testing.assert_frame_equal(out.iloc[len(df1):].reset_index(drop=True), df2)
print("PASS hai nửa của kết quả khớp df1 và df2")

# df2 khác thứ tự cột: concat vẫn ghép theo tên cột (cách numpy/loop theo vị trí sẽ sai -> chỉ kiểm cách concat)
df2_shuffled = df2[["age", "name", "student_id"]]
out = concatenateTables(df1, df2_shuffled)
pd.testing.assert_frame_equal(out, expected_example)
print("PASS df2 khác thứ tự cột -> vẫn đúng với concat")

# Bảng đầu vào không bị sửa
b1, b2 = df1.copy(), df2.copy()
for key, fn in impls.items():
    fn(df1, df2)
pd.testing.assert_frame_equal(df1, b1)
pd.testing.assert_frame_equal(df2, b2)
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: đảo thứ tự hai bảng thì kết quả khác
assert concatenateTables(df2, df1)["student_id"].tolist() == [5, 6, 1, 2, 3, 4]
print("PASS phản ví dụ: concat([df2, df1]) đảo thứ tự")

# Phản ví dụ 2: axis=1 nối ngang -> sai hình dạng
assert pd.concat([df1, df2], axis=1).shape == (4, 6)
print("PASS phản ví dụ: axis=1 cho hình dạng (4, 6), không phải (6, 3)")

PASS hai bảng rỗng
PASS hai nửa của kết quả khớp df1 và df2
PASS df2 khác thứ tự cột -> vẫn đúng với concat
PASS đầu vào không bị sửa
PASS phản ví dụ: concat([df2, df1]) đảo thứ tự
PASS phản ví dụ: axis=1 cho hình dạng (4, 6), không phải (6, 3)


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [23]:
import random

random.seed(2888)
for trial in range(200):
    n1, n2 = random.randint(0, 8), random.randint(0, 8)
    a = make([random.randint(1, 50) for _ in range(n1)],
             names=[random.choice(["An", "Binh", "Chi"]) for _ in range(n1)],
             ages=[random.randint(5, 20) for _ in range(n1)])
    b = make([random.randint(1, 50) for _ in range(n2)],
             names=[random.choice(["An", "Binh", "Chi"]) for _ in range(n2)],
             ages=[random.randint(5, 20) for _ in range(n2)])
    ref = pd.concat([a, b], ignore_index=True)
    assert len(ref) == n1 + n2
    for key, fn in impls.items():
        pd.testing.assert_frame_equal(fn(a, b), ref)
print("PASS 200 cặp bảng ngẫu nhiên - mọi cách khớp nhau")

PASS 200 cặp bảng ngẫu nhiên - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `df1.append(df2)` (hàm đã bị gỡ)

In [24]:
try:
    df1.append(df2)
except AttributeError as e:
    print("AttributeError:", e)
print("đúng:", concatenateTables(df1, df2).shape)

AttributeError: 'DataFrame' object has no attribute 'append'
đúng: (6, 3)


**Vì sao sai:** `DataFrame.append` đã bị **gỡ khỏi pandas 2.0** (và mọi bản sau, gồm pandas 3). Hãy dùng `pd.concat([df1, df2])`. Các bài/code cũ trên mạng vẫn còn `append`, nên gặp thì hãy đổi.

### Lỗi 2: Quên dấu ngoặc vuông: `pd.concat(df1, df2)`

In [25]:
try:
    pd.concat(df1, df2)
except TypeError as e:
    print("TypeError:", str(e)[:90])
print("đúng:", pd.concat([df1, df2]).shape)

TypeError: concat() takes 1 positional argument but 2 were given
đúng: (6, 3)


**Vì sao sai:** `pd.concat` nhận **một danh sách** các bảng làm tham số đầu tiên; tham số thứ hai là `axis`. Viết `pd.concat(df1, df2)` truyền `df2` vào chỗ `axis`. Luôn viết `pd.concat([df1, df2])`.

### Lỗi 3: Nhầm `axis=1` (nối ngang)

In [26]:
sai = pd.concat([df1, df2], axis=1)
print("sai  :", sai.shape, "<- 6 cột, 4 hàng, có ô NaN")
print(sai.isna().sum().to_dict())
dung = pd.concat([df1, df2], axis=0)
print("đúng :", dung.shape)

sai  : (4, 6) <- 6 cột, 4 hàng, có ô NaN
{'student_id': 2, 'name': 2, 'age': 2}
đúng : (6, 3)


**Vì sao sai:** `axis=1` ghép **theo cột** và căn theo nhãn hàng, nên hai hàng cuối của `df1` không có "bạn" bên `df2` và thành `NaN`. Bài này cần thêm **hàng**: dùng `axis=0` (mặc định).

### Lỗi 4: Quên đánh lại index

In [27]:
sai = pd.concat([df1, df2])
print("sai  : nhãn", sai.index.tolist())
print("       sai.loc[0] có", len(sai.loc[[0]]), "hàng (lẽ ra 1)")
dung = pd.concat([df1, df2], ignore_index=True)
print("đúng : nhãn", dung.index.tolist())

sai  : nhãn [0, 1, 2, 3, 0, 1]
       sai.loc[0] có 2 hàng (lẽ ra 1)
đúng : nhãn [0, 1, 2, 3, 4, 5]


**Vì sao dễ gây lỗi:** nhãn lặp `0, 1, 2, 3, 0, 1` làm `.loc[0]` trả nhiều hàng và làm so sánh với bảng mong đợi (nhãn `0..5`) bị lệch. Dùng `ignore_index=True` hoặc `reset_index(drop=True)` (đối chiếu đề xem có cần không).

### Lỗi 5: Nối hai bảng khác tên cột

In [28]:
other = df2.rename(columns={"age": "tuoi"})
mixed = pd.concat([df1, other], ignore_index=True)
print(mixed)
print("số ô NaN:", mixed.isna().sum().to_dict())

   student_id     name   age  tuoi
0           1    Mason   8.0   NaN
1           2      Ava   6.0   NaN
2           3   Taylor  15.0   NaN
3           4  Georgia  17.0   NaN
4           5      Leo   NaN   7.0
5           6     Alex   NaN   7.0
số ô NaN: {'student_id': 0, 'name': 0, 'age': 2, 'tuoi': 4}


**Vì sao sai:** `concat` ghép theo **tên cột**; `tuoi` không trùng `age` nên sinh **thêm một cột** và nhiều `NaN`, đồng thời cột `age` đổi sang `float64`. Kiểm tra `df1.columns.equals(df2.columns)` trước khi nối, hoặc dùng `join="inner"` để chỉ giữ cột chung.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "nối / gộp hai bảng cùng cột / xếp chồng / union" → nghĩ tới **`pd.concat([a, b])`**.
- Công thức nhớ: **`pd.concat([df1, df2], ignore_index=True)`**; nhiều bảng: `pd.concat([df1, df2, df3, ...])`.
- Muốn **ghép cột theo khóa chung** (kiểu SQL JOIN) thì không phải `concat` mà là **`merge`**.
- Nối ngang theo nhãn hàng: `pd.concat([a, b], axis=1)`.
- Muốn biết hàng đến từ bảng nào: `keys=[...]` hoặc thêm cột `source` trước khi nối (`a.assign(source="a")`).
- Nối **trong vòng lặp**: gom các bảng vào **danh sách** rồi `concat` **một lần** ở cuối (đừng `concat` từng cặp trong vòng lặp, sẽ rất chậm).

## 15. Biến thể của bài

1. Nối **ba** bảng trở lên: `pd.concat([df1, df2, df3], ignore_index=True)`.
2. **Nối ngang**: `pd.concat([df1, df2], axis=1)`.
3. Chỉ giữ **cột chung**: `pd.concat([a, b], join="inner")`.
4. Nối rồi **bỏ hàng trùng**: `pd.concat([df1, df2]).drop_duplicates()`.
5. Nối rồi **sắp xếp** theo `age`: `pd.concat([df1, df2]).sort_values("age")`.
6. Gắn **nhãn nguồn**: `pd.concat([df1.assign(src="df1"), df2.assign(src="df2")], ignore_index=True)`.

Ta thử vài biến thể:

In [29]:
v3 = make([9], names=["Zoe"], ages=[11])
print("1. ba bảng    :", pd.concat([df1, df2, v3], ignore_index=True).shape)
a = df1.assign(extra=1)
print("3. cột chung  :", pd.concat([a, df2], join="inner").columns.tolist())
dup = pd.concat([df1, df1]).drop_duplicates()
print("4. bỏ trùng   :", dup.shape)
print("5. sắp theo age:", pd.concat([df1, df2]).sort_values("age")["age"].tolist())
print("6. nhãn nguồn :", pd.concat([df1.assign(src="df1"), df2.assign(src="df2")], ignore_index=True)["src"].tolist())

1. ba bảng    : (7, 3)
3. cột chung  : ['student_id', 'name', 'age']
4. bỏ trùng   : (4, 3)
5. sắp theo age: [6, 7, 7, 8, 15, 17]
6. nhãn nguồn : ['df1', 'df1', 'df1', 'df1', 'df2', 'df2']


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp `iterrows` + danh sách | gom từng hàng | `O((m1+m2)·c)` (rất chậm) | `O((m1+m2)·c)` | hiểu bản chất; không dùng thực tế |
| 2. `pd.concat([..], ignore_index=True)` | nối dọc, đánh lại nhãn | `O((m1+m2)·c)` | `O((m1+m2)·c)` | **cách nộp bài** |
| 3. `concat` rồi `reset_index(drop=True)` | giữ nhãn gốc trước, đánh lại sau | `O((m1+m2)·c)` | `O((m1+m2)·c)` | cần xem nhãn gốc / `keys=` trước khi bỏ |
| 4. `to_dict("records") + ...` | nối hai danh sách từ điển | `O((m1+m2)·c)` | `O((m1+m2)·c)` | dữ liệu nhỏ, trộn với danh sách Python |
| 5. `np.concatenate` | nối mảng NumPy | `O((m1+m2)·c)` | `O((m1+m2)·c)` | chỉ để hiểu; mất kiểu cột nên phải `astype` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `pd.concat([df1, df2]).shape`, `pd.concat([df1, df2]).index.tolist()`, `pd.concat([df1, df2], ignore_index=True).index.tolist()`, `pd.concat([df1, df2], axis=1).shape`, `len(pd.concat([df2, df2]))`.

**Bài 2 (tính tay).** Hai bảng `a` có `id = [10, 20]` và `b` có `id = [30]`. Viết `id` và nhãn index của `pd.concat([a, b])`, rồi của `pd.concat([b, a], ignore_index=True)`.

**Bài 3 (điền TODO).** Hoàn thành `concat_todo(a, b)` bằng `concat`.

In [30]:
def concat_todo(a, b):
    # TODO 1: gom hai bảng vào một danh sách [a, b]
    # TODO 2: return pd.concat(danh_sach, ignore_index=True)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `concat_loop2(df1, df2)` bằng vòng lặp thuần Python (Cách 1 hoặc Cách 4) mà không nhìn code mẫu. Kiểm tra với ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `concat_many(tables)` nhận **danh sách** bảng và trả về bảng nối (xử lý cả danh sách rỗng bằng cách trả bảng rỗng có các cột `student_id`, `name`, `age`). (b) Viết `concat_dedup(df1, df2)` nối hai bảng rồi bỏ các hàng **trùng hoàn toàn**, giữ hàng đầu tiên, và đánh lại nhãn.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `(6, 3)`, `[0, 1, 2, 3, 0, 1]`, `[0, 1, 2, 3, 4, 5]`, `(4, 6)`, `4`. Bài 2: `concat` giữ nhãn gốc, `ignore_index=True` đánh lại.
- **Mức 2 (vừa):** Bài 2: `pd.concat([a, b])` -> `id [10, 20, 30]`, nhãn `[0, 1, 0]`; `pd.concat([b, a], ignore_index=True)` -> `id [30, 10, 20]`, nhãn `[0, 1, 2]`. Bài 3: hai dòng đã ghi trong comment. Bài 4: tạo `rows`, duyệt từng bảng bằng `to_dict("records")`, nối rồi `pd.DataFrame(rows, columns=...)`.
- **Mức 3 (sâu):** Bài 5(a): `pd.concat(tables, ignore_index=True)` khi `tables` không rỗng; nếu rỗng thì `pd.DataFrame(columns=[...])` (`pd.concat([])` báo `ValueError`). Bài 5(b): `pd.concat([df1, df2]).drop_duplicates().reset_index(drop=True)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert pd.concat([df1, df2]).shape == (6, 3)
assert pd.concat([df1, df2]).index.tolist() == [0, 1, 2, 3, 0, 1]
assert pd.concat([df1, df2], ignore_index=True).index.tolist() == [0, 1, 2, 3, 4, 5]
assert pd.concat([df1, df2], axis=1).shape == (4, 6)
assert len(pd.concat([df2, df2])) == 4

# Bài 2
a2 = pd.DataFrame({"id": [10, 20]})
b2 = pd.DataFrame({"id": [30]})
r = pd.concat([a2, b2])
assert r["id"].tolist() == [10, 20, 30] and r.index.tolist() == [0, 1, 0]
r = pd.concat([b2, a2], ignore_index=True)
assert r["id"].tolist() == [30, 10, 20] and r.index.tolist() == [0, 1, 2]

# Bài 3
def concat_todo_ans(a, b):
    danh_sach = [a, b]
    return pd.concat(danh_sach, ignore_index=True)

pd.testing.assert_frame_equal(concat_todo_ans(df1, df2), expected_example)

# Bài 4
def concat_loop2(df1, df2):
    rows = df1.to_dict("records") + df2.to_dict("records")
    return pd.DataFrame(rows, columns=df1.columns).astype(df1.dtypes.to_dict())

pd.testing.assert_frame_equal(concat_loop2(df1, df2), expected_example)

# Bài 5(a)
def concat_many(tables):
    if not tables:
        return pd.DataFrame({"student_id": pd.Series([], dtype="int64"),
                             "name": pd.Series([], dtype="str"),
                             "age": pd.Series([], dtype="int64")})
    return pd.concat(tables, ignore_index=True)

assert len(concat_many([df1, df2, df2])) == 8
assert len(concat_many([])) == 0 and list(concat_many([]).columns) == ["student_id", "name", "age"]

# Bài 5(b)
def concat_dedup(df1, df2):
    return pd.concat([df1, df2]).drop_duplicates().reset_index(drop=True)

r = concat_dedup(df1, df1)
pd.testing.assert_frame_equal(r, df1)
assert len(concat_dedup(df1, df2)) == 6
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (nối hai bảng cùng cột theo chiều dọc)

```python
import pandas as pd

def concatenateTables(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    return pd.concat([df1, df2], ignore_index=True)
```

**Checklist**

- [ ] Dùng `pd.concat([df1, df2])` với **danh sách** `[...]` (không `append`, không `concat(df1, df2)`)?
- [ ] `axis=0` (mặc định), không phải `axis=1`?
- [ ] Thứ tự đúng: `df1` trước, `df2` sau?
- [ ] Đã `ignore_index=True` (hoặc `reset_index(drop=True)`) nếu cần nhãn `0..n-1`; đối chiếu đề?
- [ ] Hai bảng **cùng tên cột** (không sinh cột thừa/`NaN`)?
- [ ] **Trả về** DataFrame; bảng đầu vào không bị sửa?
- [ ] Đã thử bảng rỗng ở một hoặc cả hai phía?
- [ ] Skeleton đúng `def concatenateTables(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame` (đối chiếu lại với đề gốc nếu lệch)?